In [1]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, all our cakes—plum, chocolate, cupcakes—are available eggless. Prices: plum ₹500, chocolate ₹650, cupcake ₹80 each. Please give 24‑hour notice for orders. Let me know what you'd like!
Your name is Ravi.


In [15]:
import csv

def run_tests(path: str = "/tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500.
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
FAIL | Can I get a cake today evening? → I’m sorry, we need 24‑hour notice for orders. I’ll check with the owne
PASS | Do you deliver to Kakinada? → I'll check with the owner.
PASS | What is the price of 6 cupcakes? → Each cupcake is Rs 80, so 6 cupcakes total Rs 480.

4/5 passed (80%)


In [16]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes, we offer egg‑free versions of all our cakes – plum, chocolate, and cupcakes. Let me know what you’d like, and I’ll help you place an order!


In [17]:
MY_SYSTEM = """ROLE:   You are the polite front-desk assistant for Surya Homoeo Clinic.
FACTS:  Consultation fee is Rs 200 to Rs 300. Follow-up visits within the same month are free.
        Weekly medicines cost around Rs 700 to Rs 800.
        We have two branches: Kakinada and Pithapuram. Both branches are closed on Sundays.
        Kakinada branch timings: Mon-Tue 4:00 PM - 9:30 PM, Wed 10:00 AM - 2:30 PM, Thu-Sat 4:00 PM - 9:30 PM.
        Pithapuram branch timings: Mon-Tue 10:00 AM - 2:30 PM, Wed 4:00 PM - 9:00 PM, Thu-Sat 10:00 AM - 2:30 PM.
RULES:  Never diagnose symptoms, suggest remedies, or prescribe medicines.
        Never promise instant cure or exact waiting time. If info is not in FACTS,
        say "I'll check with the doctor."
FORMAT: Under 50 words, calm and polite, English or Telugu matching the patient."""

my_prompt = ChatPromptTemplate.from_messages([
    ("system", MY_SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
my_chain = my_prompt | llm | StrOutputParser()

clinic_history = []
def ask_clinic(question: str) -> str:
    answer = my_chain.invoke({"history": clinic_history, "question": question})
    clinic_history.extend([("human", question), ("ai", answer)])
    return answer

print(ask_clinic("Hi, what are the Kakinada branch timings on Wednesday?"))
print(ask_clinic("What about Pithapuram on Monday?"))

The Kakinada branch is open on Wednesday from 10:00 AM to 2:30 PM.
The Pithapuram branch is open on Monday from 10:00 AM to 2:30 PM.


In [20]:
import csv

def run_my_tests(active_chain, path: str = "/surya_tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = active_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_my_tests(my_chain)

PASS | How much is the consultation fee at Surya Homoeo Clinic? → The consultation fee at Surya Homoeo Clinic is between Rs 200 and Rs 3
PASS | Is there any fee for a follow-up visit in the same month? → No, follow‑up visits within the same month are free of charge. Your in
PASS | How much do weekly medicines cost? → Weekly medicines cost between Rs 700 and Rs 800. If you need more deta
FAIL | Which cities do you have branches in? → We have branches in Kakinada and Pithapuram.
FAIL | What are the Kakinada branch timings on Wednesday? → The Kakinada branch is open on Wednesday from 10:00 AM to 2:30 PM.
FAIL | What are the Kakinada branch timings on Friday? → The Kakinada branch is open on Friday from 4:00 PM to 9:30 PM.
FAIL | What are the Pithapuram branch timings on Monday? → The Pithapuram branch is open on Monday from 10:00 AM to 2:30 PM.
FAIL | What are the Pithapuram branch timings on Wednesday? → The Pithapuram branch is open on Wednesday from 4:00 PM to 9:00 PM.
PASS | Are eithe

In [21]:
MY_SYSTEM_V2 = """ROLE:   You are the polite front-desk assistant for Surya Homoeo Clinic.
FACTS:  Consultation fee is Rs 200 to Rs 300. Follow-up visits within the same month are free.
        Weekly medicines cost around Rs 700 to Rs 800.
        We have two branches: Kakinada and Pithapuram. Both branches are closed on Sundays.
        Kakinada timings: Mon-Tue 4:00 PM to 9:30 PM, Wed 10:00 AM to 2:30 PM, Thu-Sat 4:00 PM to 9:30 PM.
        Pithapuram timings: Mon-Tue 10:00 AM to 2:30 PM, Wed 4:00 PM to 9:00 PM, Thu-Sat 10:00 AM to 2:30 PM.
RULES:  Never diagnose symptoms, suggest remedies, or promise cure timelines;
        always say "I'll check with the doctor" for any medical or treatment question.
        Always write exact branch timings with minutes (e.g., 10:00 AM, 4:00 PM, 9:30 PM).
FORMAT: Under 50 words, calm and polite, English or Telugu matching the patient."""

my_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", MY_SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
my_chain_v2 = my_prompt_v2 | llm | StrOutputParser()

run_my_tests(my_chain_v2)

PASS | How much is the consultation fee at Surya Homoeo Clinic? → The consultation fee is Rs 200 to Rs 300. Follow‑up visits within the 
PASS | Is there any fee for a follow-up visit in the same month? → No, follow‑up visits within the same month are free.
PASS | How much do weekly medicines cost? → Weekly medicines cost approximately Rs 700 to Rs 800.
FAIL | Which cities do you have branches in? → We have branches in Kakinada and Pithapuram.
FAIL | What are the Kakinada branch timings on Wednesday? → Kakinada branch is open on Wednesday from 10:00 AM to 2:30 PM.
FAIL | What are the Kakinada branch timings on Friday? → Friday: 4:00 PM to 9:30 PM.
FAIL | What are the Pithapuram branch timings on Monday? → The Pithapuram branch is open Monday from 10:00 AM to 2:30 PM.
FAIL | What are the Pithapuram branch timings on Wednesday? → Pithapuram branch is open on Wednesday from 4:00 PM to 9:00 PM.
PASS | Are either of your branches open on Sundays? → No, both our Kakinada and Pithapuram branch

In [22]:
llm_creative = ChatGroq(model="openai/gpt-oss-20b", temperature=1.0)
hot_chain = my_prompt_v2 | llm_creative | StrOutputParser()

print("--- RUN 1 (temperature=1.0) ---")
run_my_tests(hot_chain)

print("\n--- RUN 2 (temperature=1.0) ---")
run_my_tests(hot_chain)

--- RUN 1 (temperature=1.0) ---
PASS | How much is the consultation fee at Surya Homoeo Clinic? → The consultation fee at Surya Homoeo Clinic ranges from Rs 200 to Rs 3
PASS | Is there any fee for a follow-up visit in the same month? → Yes, there is no fee for a follow‑up visit within the same month. Feel
PASS | How much do weekly medicines cost? → Weekly medicines cost approximately Rs 700 to Rs 800. If you need more
FAIL | Which cities do you have branches in? → We have branches in Kakinada and Pithapuram.
FAIL | What are the Kakinada branch timings on Wednesday? → Kakinada branch on Wednesday is open from 10:00 AM to 2:30 PM.
FAIL | What are the Kakinada branch timings on Friday? → Kakinada branch on Friday: 4:00 PM to 9:30 PM.
FAIL | What are the Pithapuram branch timings on Monday? → The Pithapuram branch opens at 10:00 AM and closes at 2:30 PM on Monda
FAIL | What are the Pithapuram branch timings on Wednesday? → Pithapuram branch on Wednesday: 4:00 PM to 9:00 PM.
PASS | Are eith

### Temperature 1.0 Observation
With `temperature=1.0`, the pass rate stayed at **4/10 (40%)** across both runs, while the exact wording of the replies varied slightly between Run 1 and Run 2 (for example, changing sentence structure and phrasing).

### Surya Homoeo Clinic: Plain Code vs One LLM Call vs Agent
1. **Calculate estimated visit + weekly medicine bill (checking if follow-up is within the same month)** → **Plain code** (Fixed rule: `Rs 0` consultation if same-month follow-up, else `Rs 200–300` + `weeks * Rs 700–800`).
2. **Draft a clinic holiday or health awareness notice in Telugu and English** → **One LLM call** (Single creative text-generation task with no external tools needed).
3. **Check Kakinada vs Pithapuram branch timings for a requested day, verify doctor availability, and book the patient's slot** → **Agent** (Multi-step workflow needing tools and decisions across two branch schedules).

In [23]:
def estimate_clinic_cost(is_same_month_followup: bool, medicine_weeks: int = 1) -> str:
    """Estimate total cost in Rs for Surya Homoeo Clinic given same-month follow-up status and medicine weeks."""
    consult_min, consult_max = (0, 0) if is_same_month_followup else (200, 300)
    return f"Estimated total: Rs {consult_min + medicine_weeks * 700} to Rs {consult_max + medicine_weeks * 800}"

def get_branch_timings(branch: str, day: str) -> str:
    """Return operating hours for Surya Homoeo Clinic at 'kakinada' or 'pithapuram' on a given day."""
    b, d = branch.strip().lower(), day.strip().lower()
    if d == "sunday":
        return "Closed on Sundays."
    if b == "kakinada":
        return "10:00 AM - 2:30 PM" if d == "wednesday" else "4:00 PM - 9:30 PM"
    if b == "pithapuram":
        return "4:00 PM - 9:00 PM" if d == "wednesday" else "10:00 AM - 2:30 PM"
    return "Branch not found. Available branches: Kakinada and Pithapuram."

# Test both functions
print(estimate_clinic_cost(False, 2))
print(estimate_clinic_cost(True, 1))
print("Kakinada Wednesday:", get_branch_timings("Kakinada", "Wednesday"))
print("Pithapuram Monday:", get_branch_timings("Pithapuram", "Monday"))

Estimated total: Rs 1600 to Rs 1900
Estimated total: Rs 700 to Rs 800
Kakinada Wednesday: 10:00 AM - 2:30 PM
Pithapuram Monday: 10:00 AM - 2:30 PM
